# 01 — TF-IDF con il titolo (Versione 4)

Il TF-IDF + regressione logistica della Versione 2 (5 livelli, testate di solo training con il loro peso, classi
bilanciate sui pesi, test sulle 304 testate valutate, fold per gruppo), con il **titolo** come blocco di parole a
parte (`TITOLO = True`): un secondo TF-IDF sui soli titoli, affiancato a quello sull'articolo. È il confronto onesto
per `02_transformer_titolo`: anche il TF-IDF riceve il titolo, ma lo conta, non lo confronta con l'articolo.

Con `SVILUPPO = True` (prove preliminari) si usa solo la divisione di sviluppo (`comune.divisione_sviluppo`, il fold 1
non si guarda mai) e si salvano le metriche sulla validazione. Testata = media delle probabilità dei suoi articoli.

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, RESULTS, SEED, carica, divisione_sviluppo, metriche, per_testata, pesi_bilanciati, registratore

In [ ]:
NOME = "feature_titolo"
TITOLO = True
C_GRIGLIA = ""             # "" = C = 1; "0.1,1,10,100" = C scelto in validazione interna (solo training)
SVILUPPO = False
N_JOBS = 5
PROVA = False

In [ ]:
base = RESULTS / ("prova" if PROVA else "sviluppo" if SVILUPPO else "")
OUT = base / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(base / f"avanzamento_{NOME}.log")
art = carica(PROVA)
art["titolo"] = art.titolo.fillna("").astype(str)
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate, titolo {TITOLO}, C {C_GRIGLIA or 1}, sviluppo {SVILUPPO} ===")


def matrici(a, b):
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=5 if not PROVA else 2, max_df=0.9, sublinear_tf=True, max_features=100_000, strip_accents="unicode")
    Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
    if TITOLO:            # un TF-IDF a parte sui titoli, affiancato
        w = TfidfVectorizer(ngram_range=(1, 2), min_df=3 if not PROVA else 1, sublinear_tf=True, max_features=30_000, strip_accents="unicode")
        Xa, Xb = sp.hstack([Xa, w.fit_transform(a.titolo)]).tocsr(), sp.hstack([Xb, w.transform(b.titolo)]).tocsr()
    return Xa, Xb


def adatta_predici(a, b, c):
    Xa, Xb = matrici(a, b)
    clf = LogisticRegression(C=c, max_iter=2000, random_state=SEED).fit(Xa, a.label, sample_weight=pesi_bilanciati(a.label, a.peso))
    P = np.zeros((len(b), 5)); P[:, clf.classes_] = clf.predict_proba(Xb)
    L = np.full((len(b), 5), np.nan); L[:, clf.classes_] = clf.decision_function(Xb)
    return P, L


def tabella(b, P, L):
    t = pd.DataFrame({"article_id": b.article_id.values, "domain": b.domain.values, "fold": b.fold_gruppo.values, "y_true": b.label.values})
    for k in range(5):
        t[f"logit_l{k + 1}"] = L[:, k]
    for k, c in enumerate(CLASSI):
        t[c] = P[:, k]
    t["y_pred"] = P.argmax(1)
    return t


def scegli_C(tr):
    griglia = [float(x) for x in C_GRIGLIA.split(",")]
    t = tr.drop_duplicates("domain").reset_index(drop=True); punti = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(t, t.label, t.gruppo):
        a = tr[tr.domain.isin(t.domain[itr])]; b = tr[tr.domain.isin(t.domain[iva]) & tr.valutazione]
        for c in griglia:
            P, L = adatta_predici(a, b, c)
            punti[c].append(metriche(tabella(b, P, L))["testata_media"]["f1_macro"])
    medie = {c: float(np.mean(s)) for c, s in punti.items()}
    return max(medie, key=medie.get), medie


def un_fold(f):
    t0 = time.time()
    if SVILUPPO:
        i_tr, i_va = divisione_sviluppo(art); tr, te = art.loc[i_tr], art.loc[i_va]
    else:
        tr, te = art[art.fold_gruppo != f], art[(art.fold_gruppo == f) & art.valutazione]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    P, L = adatta_predici(tr, te, C)
    return f, tabella(te, P, L), C, medie, time.time() - t0


folds = [1] if SVILUPPO else sorted(art.fold_gruppo.unique())
parti = []
for f, t, C, medie, sec in Parallel(n_jobs=min(N_JOBS, len(folds)))(delayed(un_fold)(f) for f in folds):
    parti.append(t); m = metriche(t)["testata_media"]
    registra(f"{'sviluppo' if SVILUPPO else f'fold {f}'}: C {C}" + (f" (interne {', '.join(f'{k}: {x:.3f}' for k, x in medie.items())})" if medie else "")
             + f"; F1 {m['f1_macro']:.3f}, acc {m['accuratezza']:.3f}, bin {m['bin_accuratezza']:.3f} ({sec / 60:.0f} min)")
pred = pd.concat(parti).sort_values("article_id")
pred.to_csv(OUT / "articoli.csv", index=False)
m = metriche(pred); json.dump(m, open(OUT / ("metriche_sviluppo.json" if SVILUPPO else "metriche.json"), "w"), indent=1)
(OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
registra(f"=== {NOME} FINITO: F1 {m['testata_media']['f1_macro']:.3f}, acc {m['testata_media']['accuratezza']:.3f}, bin {m['testata_media']['bin_accuratezza']:.3f} ===")
pd.DataFrame(m).T[["accuratezza", "f1_macro", "bin_accuratezza", "tolerant_1"]].round(3)